## SQL Databases and LangChain
SQL databases are structured collections of data that use SQL (Structured Query Language) for managing and manipulating the data. They are widely used in various applications for storing and retrieving information efficiently. LangChain provides tools to interact with SQL databases, allowing you to chunk large datasets into manageable pieces for processing and analysis.

In [1]:
import sqlite3
import os 

os.makedirs("./data/db", exist_ok=True)

In [2]:
# create a new sample database
conn = sqlite3.connect("./data/db/company.db")
cursor = conn.cursor() 

In [3]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS employees 
(id INTEGER PRIMARY KEY, name TEXT, position TEXT, salary REAL)
''')

In [4]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS projects 
(id INTEGER PRIMARY KEY, name TEXT, start_date TEXT, end_date TEXT, lead_id INTEGER,
FOREIGN KEY (lead_id) REFERENCES employees(id))
''')

In [5]:
employees = [
    (1, 'Alice Johnson', 'Software Engineer', 90000),
    (2, 'Bob Smith', 'Data Scientist', 95000),
    (3, 'Charlie Brown', 'Product Manager', 105000),
]

projects = [
    (1, 'Project Alpha', '2023-01-15', '2023-06-30', 1),
    (2, 'Project Beta', '2023-03-01', '2023-12-31', 2),
    (3, 'Project Gamma', '2023-05-20', '2024-05-20', 3),
]

cursor.executemany('INSERT OR IGNORE INTO employees VALUES (?, ?, ?, ?)', employees)
cursor.executemany('INSERT OR IGNORE INTO projects VALUES (?, ?, ?, ?, ?)', projects)

In [6]:
cursor.execute('SELECT * FROM employees')

In [7]:
conn.commit()

In [8]:
conn.close()

## Databse content extraction example

In [10]:
from langchain_community.utilities import SQLDatabase 
from langchain_community.document_loaders import SQLDatabaseLoader

In [13]:
# Method 1: SQLDatabase utility
db = SQLDatabase.from_uri("sqlite:///./data/db/company.db")

# Get the database information
print(db.get_table_info(["employees"]))
print(db.get_table_info(["projects"]))
print(db.get_table_names())


CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	position TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	position	salary
1	Alice Johnson	Software Engineer	90000.0
2	Bob Smith	Data Scientist	95000.0
3	Charlie Brown	Product Manager	105000.0
*/

CREATE TABLE projects (
	id INTEGER, 
	name TEXT, 
	start_date TEXT, 
	end_date TEXT, 
	lead_id INTEGER, 
	PRIMARY KEY (id), 
	FOREIGN KEY(lead_id) REFERENCES employees (id)
)

/*
3 rows from projects table:
id	name	start_date	end_date	lead_id
1	Project Alpha	2023-01-15	2023-06-30	1
2	Project Beta	2023-03-01	2023-12-31	2
3	Project Gamma	2023-05-20	2024-05-20	3
*/
['employees', 'projects']


In [14]:
# Method 2: Custom SQL to Documents conversion

from typing import List
from langchain_core.documents import Document

def sql_to_documents(db:str) -> List[Document]:
    # Convert SQL database to documents
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    documents = []
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table_name in tables:
        table_name = table_name[0]
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        column_names = [col[1] for col in columns]

        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        for row in rows:
            content = f"Table: {table_name}\n"
            for col_name, value in zip(column_names, row):
                content += f"{col_name}: {value}\n"
            documents.append(Document(page_content=content.strip()))
    conn.close()
    return documents

In [15]:
sql_to_documents("./data/db/company.db")

[Document(metadata={}, page_content='Table: employees\nid: 1\nname: Alice Johnson\nposition: Software Engineer\nsalary: 90000.0'),
 Document(metadata={}, page_content='Table: employees\nid: 2\nname: Bob Smith\nposition: Data Scientist\nsalary: 95000.0'),
 Document(metadata={}, page_content='Table: employees\nid: 3\nname: Charlie Brown\nposition: Product Manager\nsalary: 105000.0'),
 Document(metadata={}, page_content='Table: projects\nid: 1\nname: Project Alpha\nstart_date: 2023-01-15\nend_date: 2023-06-30\nlead_id: 1'),
 Document(metadata={}, page_content='Table: projects\nid: 2\nname: Project Beta\nstart_date: 2023-03-01\nend_date: 2023-12-31\nlead_id: 2'),
 Document(metadata={}, page_content='Table: projects\nid: 3\nname: Project Gamma\nstart_date: 2023-05-20\nend_date: 2024-05-20\nlead_id: 3')]

In [20]:
  from langchain_text_splitters import RecursiveCharacterTextSplitter

  documents = sql_to_documents("./data/db/company.db")
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
  texts = text_splitter.split_documents(documents)
  print(f"Number of chunks: {len(texts)}")
  for i, chunk in enumerate(texts[:3]):
      print(f"\n--- Chunk {i+1} ---\n{chunk.page_content}")

Number of chunks: 6

--- Chunk 1 ---
Table: employees
id: 1
name: Alice Johnson
position: Software Engineer
salary: 90000.0

--- Chunk 2 ---
Table: employees
id: 2
name: Bob Smith
position: Data Scientist
salary: 95000.0

--- Chunk 3 ---
Table: employees
id: 3
name: Charlie Brown
position: Product Manager
salary: 105000.0
